In [2]:
%pip install -q sentence-transformers qdrant-client openai

In [3]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from openai import OpenAI
import getpass

In [5]:
data = [
    {
        "question": "What is RAG?",
        "answer": (
            "Retrieval-Augmented Generation (RAG) retrieves relevant "
            "information from a knowledge source and provides it to a "
            "language model to help generate an answer."
        )
    },
    {
        "question": "What are embeddings?",
        "answer": (
            "Embeddings are numerical vectors that represent the meaning "
            "of text and help computers find semantically similar content."
        )
    },
    {
        "question": "What is a vector database?",
        "answer": (
            "A vector database stores vector embeddings and supports "
            "similarity searches to retrieve relevant information."
        )
    },
    {
        "question": "What is a language model?",
        "answer": (
            "A language model is an AI model trained to understand "
            "and generate human language."
        )
    },
    {
        "question": "Why is RAG useful?",
        "answer": (
            "RAG provides a language model with relevant external "
            "information, helping it answer questions using a specific "
            "knowledge source."
        )
    }
]

print("Number of records:", len(data))

Number of records: 5


In [6]:
documents = [
    {
        "text": f"Question: {item['question']} Answer: {item['answer']}",
        "question": item["question"],
        "answer": item["answer"]
    }
    for item in data
]

print(documents[0]["text"])

Question: What is RAG? Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.


In [7]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["text"] for doc in documents]

embeddings = encoder.encode(
    texts,
    convert_to_numpy=True
)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of embeddings: 5
Vector dimensions: 384


In [8]:
qdrant = QdrantClient(":memory:")

collection_name = "my_rag_knowledge"

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

points = [
    models.PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload=documents[idx]
    )
    for idx in range(len(documents))
]

qdrant.upload_points(
    collection_name=collection_name,
    points=points,
    wait=True
)

print("Knowledge base created successfully!")

Knowledge base created successfully!


In [9]:
def retrieve_documents(question, limit=3):
    query_embedding = encoder.encode(question).tolist()

    results = qdrant.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=limit,
        with_payload=True
    ).points

    return [
        {
            "text": result.payload["text"],
            "score": result.score
        }
        for result in results
    ]


question = "Why do we need RAG?"

search_results = retrieve_documents(question)

for result in search_results:
    print("Score:", round(result["score"], 3))
    print(result["text"])
    print("-" * 50)

Score: 0.54
Question: Why is RAG useful? Answer: RAG provides a language model with relevant external information, helping it answer questions using a specific knowledge source.
--------------------------------------------------
Score: 0.518
Question: What is RAG? Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.
--------------------------------------------------
Score: -0.011
Question: What are embeddings? Answer: Embeddings are numerical vectors that represent the meaning of text and help computers find semantically similar content.
--------------------------------------------------


In [10]:

from openai import OpenAI
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/"
)

print("API client is ready!")

Enter your Gemini API key: ··········
API client is ready!


In [12]:
def answer_question(question):
    # 1. Retrieve relevant documents
    results = retrieve_documents(question, limit=3)

    # 2. Prepare context for the language model
    context = "\n\n".join(
        result["text"] for result in results
    )

    # 3. Generate an answer using the retrieved context
    completion = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant. "
                    "Answer using only the provided context. "
                    "If the context does not contain the answer, "
                    "say that the information is not available."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ],
        temperature=0
    )

    return completion.choices[0].message.content


question = "Why do we need RAG?"
answer = answer_question(question)

print("Question:", question)
print("Answer:", answer)

Question: Why do we need RAG?
Answer: Based on the provided context, RAG is useful/needed because it provides a language model with relevant external information, helping it answer questions using a specific knowledge source.


In [16]:
test_questions = [
    "What are embeddings?",
    "What is a vector database?",
    "What is a language model?",
    "What is RAG?"
]

for question in test_questions:
    print("Question:", question)
    print("Answer:", answer_question(question))
    print("-" * 60)

Question: What are embeddings?
Answer: Embeddings are numerical vectors that represent the meaning of text and help computers find semantically similar content.
------------------------------------------------------------
Question: What is a vector database?
Answer: A vector database stores vector embeddings and supports similarity searches to retrieve relevant information.
------------------------------------------------------------
Question: What is a language model?
Answer: A language model is an AI model trained to understand and generate human language.
------------------------------------------------------------
Question: What is RAG?
Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.
------------------------------------------------------------
